# Notebook 03 — Exploratory Data Analysis

**Maternal & Infant Mortality Risk Prediction Using Data Mining Techniques**

> ⚠️ **SYNTHETIC — generated for academic demonstration. Not real clinical data.**

---

This notebook produces **8+ labelled figures** saved to `reports/figures/`.  
Each figure is followed by a one-paragraph interpretation.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

# Plotting style
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)

FIGURES = os.path.join('..', 'reports', 'figures')
os.makedirs(FIGURES, exist_ok=True)

# Load raw synthetic data
df = pd.read_csv(os.path.join('..', 'data', 'raw', 'synthetic_maternal_infant_risk.csv'))
print(f"Shape: {df.shape}")
print('Imports OK')

## Figure 1 — Class Distribution (Risk Label)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['risk_label'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # green for LOW, red for HIGH
counts.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('SYNTHETIC — Class Distribution of Risk Label', fontweight='bold')
ax.set_xlabel('Risk Label')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Annotate counts
for i, v in enumerate(counts):
    ax.text(i, v + 30, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig01_class_distribution.png'), dpi=150)
plt.show()
print('Saved: fig01_class_distribution.png')

**Interpretation:** The class distribution shows the proportion of HIGH-RISK vs LOW-RISK pregnancies in the synthetic dataset. While the dataset was designed with clinically plausible risk factors, the Gaussian noise injection and threshold at score ≥ 5 produce a class split that reflects realistic epidemiological patterns. Any imbalance will be addressed via SMOTE during model training to prevent the classifier from being biased toward the majority class.

## Figure 2 — Mother's Age vs Risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram by risk
for label, color in zip(['LOW-RISK', 'HIGH-RISK'], ['#2ecc71', '#e74c3c']):
    subset = df[df['risk_label'] == label]
    axes[0].hist(subset['mother_age'], bins=20, alpha=0.6, label=label, color=color, edgecolor='black')
axes[0].set_title('SYNTHETIC — Age Distribution by Risk Level', fontweight='bold')
axes[0].set_xlabel('Mother Age (years)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Boxplot
sns.boxplot(data=df, x='risk_label', y='mother_age', ax=axes[1],
            palette={'LOW-RISK': '#2ecc71', 'HIGH-RISK': '#e74c3c'})
axes[1].set_title('SYNTHETIC — Age Boxplot by Risk', fontweight='bold')
axes[1].set_xlabel('Risk Label')
axes[1].set_ylabel('Mother Age (years)')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig02_age_vs_risk.png'), dpi=150)
plt.show()
print('Saved: fig02_age_vs_risk.png')

**Interpretation:** Maternal age is a known risk factor at both extremes: adolescent pregnancies (< 18 years) carry higher obstetric complications, while advanced maternal age (> 35) increases chromosomal and gestational risks. The histogram shows that HIGH-RISK cases are slightly more prevalent at the tails of the age distribution, which aligns with the +2 points assigned by the risk score rule for mothers below 18 or above 35.

## Figure 3 — Hemoglobin Distribution by Risk

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, color in zip(['LOW-RISK', 'HIGH-RISK'], ['#2ecc71', '#e74c3c']):
    subset = df[df['risk_label'] == label]
    sns.kdeplot(subset['hemoglobin_level'], ax=ax, label=label, color=color,
                fill=True, alpha=0.3, linewidth=2)

ax.axvline(x=11, color='orange', linestyle='--', label='Anemia threshold (11 g/dL)')
ax.axvline(x=7, color='red', linestyle='--', label='Severe anemia (7 g/dL)')
ax.set_title('SYNTHETIC — Hemoglobin Level Distribution by Risk', fontweight='bold')
ax.set_xlabel('Hemoglobin (g/dL)')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig03_hemoglobin_by_risk.png'), dpi=150)
plt.show()
print('Saved: fig03_hemoglobin_by_risk.png')

**Interpretation:** Hemoglobin level is one of the strongest predictors of pregnancy risk. HIGH-RISK cases are visibly shifted toward lower hemoglobin values, with a concentration below the 11 g/dL anemia threshold. Severe anemia (< 7 g/dL) contributes 4 points to the risk score — nearly enough to cross the HIGH-RISK threshold alone. This pattern mirrors WHO findings that anemia significantly increases maternal and perinatal mortality.

## Figure 4 — Blood Pressure vs Risk (Systolic)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Systolic BP
sns.violinplot(data=df, x='risk_label', y='blood_pressure_systolic', ax=axes[0],
               palette={'LOW-RISK': '#2ecc71', 'HIGH-RISK': '#e74c3c'}, inner='box')
axes[0].axhline(y=140, color='orange', linestyle='--', alpha=0.7, label='Hypertension (140)')
axes[0].set_title('SYNTHETIC — Systolic BP by Risk', fontweight='bold')
axes[0].set_ylabel('Systolic BP (mmHg)')
axes[0].legend()

# Diastolic BP
sns.violinplot(data=df, x='risk_label', y='blood_pressure_diastolic', ax=axes[1],
               palette={'LOW-RISK': '#2ecc71', 'HIGH-RISK': '#e74c3c'}, inner='box')
axes[1].axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='High diastolic (90)')
axes[1].set_title('SYNTHETIC — Diastolic BP by Risk', fontweight='bold')
axes[1].set_ylabel('Diastolic BP (mmHg)')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig04_bp_vs_risk.png'), dpi=150)
plt.show()
print('Saved: fig04_bp_vs_risk.png')

**Interpretation:** Elevated blood pressure is a cardinal sign of pregnancy-induced hypertension and pre-eclampsia. The violin plots show that HIGH-RISK pregnancies have a noticeably higher median systolic BP and a longer upper tail extending above 140 mmHg. This is consistent with the risk rule awarding 3 points for systolic BP ≥ 140 and 5 points for ≥ 160. The diastolic BP shows a correlated pattern due to the synthetic correlation injected between systolic and diastolic values.

## Figure 5 — ANC Visits vs Risk

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x='antenatal_care_visits', hue='risk_label', ax=ax,
              palette={'LOW-RISK': '#2ecc71', 'HIGH-RISK': '#e74c3c'})
ax.axvline(x=3.5, color='red', linestyle='--', alpha=0.7, label='WHO min (4 visits)')
ax.set_title('SYNTHETIC — ANC Visit Count by Risk Level', fontweight='bold')
ax.set_xlabel('Number of Antenatal Care Visits')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig05_anc_visits_vs_risk.png'), dpi=150)
plt.show()
print('Saved: fig05_anc_visits_vs_risk.png')

**Interpretation:** The WHO recommends a minimum of 4 antenatal care (ANC) visits during pregnancy. Women with fewer than 4 visits receive +2 risk points in our scoring system. The count plot shows that HIGH-RISK pregnancies are disproportionately concentrated among women with 0–3 ANC visits (left of the red line), while LOW-RISK pregnancies dominate at 5+ visits. This reflects the well-documented association between inadequate ANC and adverse maternal-infant outcomes in NFHS-5 data.

## Figure 6 — Correlation Heatmap

In [ ]:
# Encode target for correlation
df_corr = df.copy()
df_corr['risk_numeric'] = (df_corr['risk_label'] == 'HIGH-RISK').astype(int)

numeric_cols = ['mother_age', 'family_income', 'blood_pressure_systolic',
                'blood_pressure_diastolic', 'hemoglobin_level', 'gestational_age',
                'birth_weight', 'antenatal_care_visits', 'risk_numeric']

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_corr[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('SYNTHETIC — Correlation Heatmap (Numeric Features + Risk)', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig06_correlation_heatmap.png'), dpi=150)
plt.show()
print('Saved: fig06_correlation_heatmap.png')

**Interpretation:** The correlation heatmap reveals the linear relationships between numeric features and the risk label. As expected, `hemoglobin_level` shows a negative correlation with risk (lower hemoglobin → higher risk), while `blood_pressure_systolic` is positively correlated. The strong positive correlation between systolic and diastolic BP (r ≈ 0.6–0.8) reflects both physiological reality and the synthetic correlation we injected. `gestational_age` and `birth_weight` also show expected negative correlations with risk, as preterm births and low birth weight are major risk factors.

## Figure 7 — Birth Weight Boxplot by Risk

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x='risk_label', y='birth_weight', ax=ax,
            palette={'LOW-RISK': '#2ecc71', 'HIGH-RISK': '#e74c3c'})
ax.axhline(y=2.5, color='orange', linestyle='--', alpha=0.7, label='LBW threshold (2.5 kg)')
ax.axhline(y=1.5, color='red', linestyle='--', alpha=0.7, label='VLBW threshold (1.5 kg)')
ax.set_title('SYNTHETIC — Birth Weight by Risk Level', fontweight='bold')
ax.set_ylabel('Birth Weight (kg)')
ax.set_xlabel('Risk Label')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig07_birthweight_boxplot.png'), dpi=150)
plt.show()
print('Saved: fig07_birthweight_boxplot.png')

**Interpretation:** Birth weight is a critical indicator of neonatal survival. The boxplot clearly shows that HIGH-RISK pregnancies have a substantially lower median birth weight and more outliers below the 2.5 kg low-birth-weight (LBW) threshold. This is consistent with the risk score assigning 3 points for LBW and 5 points for very low birth weight (< 1.5 kg). The correlation between preterm delivery (lower gestational age) and lower birth weight is also reflected here.

## Figure 8 — Urban/Rural & Education Breakdowns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Urban / Rural
ct1 = pd.crosstab(df['residence_urban_rural'], df['risk_label'], normalize='index') * 100
ct1.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('SYNTHETIC — Risk by Residence', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xlabel('Residence')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title='Risk')

# Education level
edu_order = ['No Education', 'Primary', 'Secondary', 'Higher']
ct2 = pd.crosstab(df['education_level'], df['risk_label'], normalize='index') * 100
ct2 = ct2.reindex(edu_order)
ct2.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_title('SYNTHETIC — Risk by Education Level', fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Education Level')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15)
axes[1].legend(title='Risk')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig08_urban_rural_education.png'), dpi=150)
plt.show()
print('Saved: fig08_urban_rural_education.png')

**Interpretation:** The socioeconomic breakdown reveals two important patterns. First, rural women have a marginally higher proportion of HIGH-RISK pregnancies compared to urban women, reflecting reduced access to healthcare facilities and the higher probability of home delivery coded into the synthetic data. Second, women with no education or only primary education show higher risk rates than those with secondary or higher education. This aligns with NFHS-5 findings that education level is a strong proxy for health literacy, nutritional awareness, and healthcare utilisation in India.

## Figure 9 (Bonus) — Pregnancy Complications by Risk

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ct3 = pd.crosstab(df['pregnancy_complications'], df['risk_label'])
ct3.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax.set_title('SYNTHETIC — Pregnancy Complications by Risk Level', fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Complication Severity')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Risk')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig09_complications_by_risk.png'), dpi=150)
plt.show()
print('Saved: fig09_complications_by_risk.png')

**Interpretation:** Pregnancy complications are among the most powerful risk indicators. Women with severe complications are overwhelmingly classified as HIGH-RISK, which is expected given that severe complications contribute 4 points to the risk score — nearly reaching the threshold on their own. Even mild complications show a noticeable increase in risk proportion compared to women with no complications, validating the 1-point contribution in the risk rule.

## Summary

Nine EDA figures have been generated and saved to `reports/figures/`:

| # | Figure | Key Insight |
|---|--------|-------------|
| 1 | Class distribution | Shows HIGH/LOW class balance |
| 2 | Age vs risk | Extremes of age associate with higher risk |
| 3 | Hemoglobin by risk | Lower Hb strongly predicts HIGH-RISK |
| 4 | BP vs risk | Hypertension drives risk classification |
| 5 | ANC visits vs risk | Fewer visits → higher risk |
| 6 | Correlation heatmap | Hb and BP are strongest linear correlates |
| 7 | Birth weight boxplot | LBW and VLBW concentrated in HIGH-RISK |
| 8 | Urban/Rural & Education | Socioeconomic disparities in risk |
| 9 | Complications | Severe complications dominate HIGH-RISK |

**Next → Notebook 04: Model Building**